In [33]:
import numpy as np
import torch as th
from PIL import Image
import json, glob, os

method = ["hou21_rotate", "hou22_rotate", "iclight_rotate_512x512", "ours_256_DiFaReli", "ours_difareli++_oneshot_rotate_tomax"]
sample = json.load(open("./selected_rotate_RT.json", "r"))
meta = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/RotateSH_figures/ffhq_rotateSH_with_baseline.json", "r"))
os.makedirs("./rotateSH_figure/axis=2/", exist_ok=True)

for pid, dat in sample['pair'].items():
    src = dat['src']
    dst = dat['dst']
    if "frames" not in dat:
        continue
    frames_idx = dat['frames']

    res_img = {}
    for m in method:
        meta_dat = meta[m]
        img_dir = meta_dat['img_dir']
        itp_method = meta_dat['itp_method']
        diff_step = meta_dat['diff_step']
        n_frame_tmp = meta_dat['n_frame']

        img_list = []
        for fid in frames_idx:
            img_list.append(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame{fid}.png')

        if len(glob.glob(f'{img_dir}/src={src}/dst={dst}/{itp_method}_{diff_step}/n_frames={n_frame_tmp}/res_frame*.png')) == 0:
            res_img[m] = [np.zeros((256, 256, 3)) for _ in frames_idx]
        else:
            res_img[m] = [Image.open(f) for f in img_list]
    
    # Concatenate images horizontally
    for m in method:
        res_img[m] = np.concatenate(res_img[m], axis=1)
    # Concatenate images vertically
    res_img = np.concatenate([res_img[m] for m in method], axis=0).astype(np.uint8)
    res_img = Image.fromarray(res_img)
    res_img.save(f'./rotateSH_figure/axis=2/{pid}.png')
        






